In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../apps/api"))

import logging
from livewell.pipeline.replay import replay_signals
from livewell.ingestion.constants import INSTRUMENTS

logging.basicConfig(level=logging.WARNING)

BUCKET = "livewell-data-prod"
ENV = "prod"

print(f"Setup complete. {len(INSTRUMENTS)} instruments to replay.")

In [ ]:
total_written = 0
total_skipped = 0
total_failed = []

for inst in INSTRUMENTS:
    result = replay_signals(
        instruments=[inst["s3_key"]],
        env=ENV,
        bucket=BUCKET,
    )
    total_written += result["written"]
    total_skipped += result["skipped"]
    total_failed.extend(result["failed"])
    status = "\u2713" if not result["failed"] else "\u2717"
    print(f"{status} {inst['s3_key']:<12} written={result['written']}, skipped={result['skipped']}, failed={len(result['failed'])}")

print()
print("=" * 50)
print(f"Total written: {total_written}")
print(f"Total skipped: {total_skipped}")
print(f"Total failed:  {len(total_failed)}")
if total_failed:
    print(f"Failed IDs: {total_failed[:10]}{'...' if len(total_failed) > 10 else ''}")

In [ ]:
import boto3
dynamodb = boto3.resource("dynamodb", region_name="us-west-1")
table = dynamodb.Table(f"livewell-signals-{ENV}")
count = table.scan(Select="COUNT")["Count"]
print(f"Total signals in DynamoDB: {count}")

# Sample a replay record
resp = table.scan(
    FilterExpression=boto3.dynamodb.conditions.Attr("run_id").eq("replay"),
    Limit=1,
)
if resp["Items"]:
    sample = resp["Items"][0]
    print(f"\nSample replay record:")
    print(f"  signal_id: {sample['signal_id']}")
    print(f"  direction: {sample['direction']}")
    print(f"  signal_valid: {sample['signal_valid']}")
    print(f"  score: {sample.get('score')}")
    print(f"  run_id: {sample['run_id']}")